<a href="https://colab.research.google.com/github/atortosalopez/trabajo-final-sentimientos/blob/main/notebooks/03_deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow pandas scikit-learn

In [2]:
import pandas as pd

from google.colab import files
files.upload()

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])
X_val = tokenizer.texts_to_sequences(val_df["text"])
X_test = tokenizer.texts_to_sequences(test_df["text"])

X_train = pad_sequences(X_train, maxlen=max_len)
X_val = pad_sequences(X_val, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [4]:
y_train = train_df["label"]
y_val = val_df["label"]
y_test = test_df["label"]

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/5
234/234 ━━━━━━━━━━━━━━━━━━━━ 20s 74ms/step - accuracy: 0.6684 - loss: 0.5906 - val_accuracy: 0.7386 - val_loss: 0.5325
Epoch 2/5
234/234 ━━━━━━━━━━━━━━━━━━━━ 17s 59ms/step - accuracy: 0.8850 - loss: 0.2844 - val_accuracy: 0.7573 - val_loss: 0.5551
Epoch 3/5
234/234 ━━━━━━━━━━━━━━━━━━━━ 13s 58ms/step - accuracy: 0.9593 - loss: 0.1120 - val_accuracy: 0.7498 - val_loss: 0.6848
Epoch 4/5
234/234 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - accuracy: 0.9847 - loss: 0.0460 - val_accuracy: 0.7480 - val_loss: 0.8380
Epoch 5/5
234/234 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.9925 - loss: 0.0277 - val_accuracy: 0.7323 - val_loss: 1.1890


In [7]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.765625
Precision: 0.7456647398843931
Recall: 0.80625
F1: 0.7747747747747747

Classification report:

              precision    recall  f1-score   support

           0       0.79      0.72      0.76       800
           1       0.75      0.81      0.77       800

    accuracy                           0.77      1600
   macro avg       0.77      0.77      0.77      1600
weighted avg       0.77      0.77      0.77      1600



In [9]:
test_df["pred_dl"] = y_pred

errors = test_df[test_df["label"] != test_df["pred_dl"]]

print(errors[["text", "label", "pred_dl"]].head(10))

                                                 text  label  pred_dl
2   it seems like i have been waiting my whole lif...      1        0
5   as a witness to several greek-american wedding...      1        0
13  rodriguez does a splendid job of racial profil...      1        0
16  this remake gets all there is to get out of a ...      1        0
22  wimps out by going for that pg-13 rating , so ...      0        1
23  it is far from the worst , thanks to the topic...      1        0
24  returning aggressively to his formula of dimwi...      0        1
25  it pulls the rug out from under you , just whe...      1        0
33  the story is virtually impossible to follow he...      1        0
38  i'd be hard pressed to think of a film more cl...      0        1


In [10]:
test_df.to_csv("corpus_pred_dl.csv", index=False)